In [1]:
import os
import glob
import pandas as pd
import numpy as np

# ===================== 你只需要改这里 =====================
IN_DIR = "./"        # 你的测评结果csv所在文件夹
OUT_DIR = "./results"      # 输出每个数据集表格的文件夹
PATTERN = "*.csv"                        # 文件匹配模式，如 "ModuCell_*.csv" 或 "*.csv"
DIGITS = 2                               # 保留小数位
# =========================================================

NON_METRIC_COLS = {"software", "dataset", "ID", "id", "fold", "seed"}

def parse_software_dataset_from_filename(fp: str):
    """
    从文件名 Software_Dataset.csv 解析 software 和 dataset
    默认用最后一个 '_' 分割：software, dataset = name.rsplit('_', 1)
    """
    base = os.path.basename(fp)
    name = os.path.splitext(base)[0]
    if "_" not in name:
        return None, None
    software, dataset = name.rsplit("_", 1)
    return software, dataset

def format_mean_sd(mean_val, sd_val, digits=4):
    if pd.isna(mean_val):
        return ""
    if pd.isna(sd_val):
        return f"{mean_val:.{digits}f} (nan)"
    return f"{mean_val:.{digits}f} ({sd_val:.{digits}f})"

def summarize_one_file(fp: str, digits: int):
    df = pd.read_csv(fp)

    # software/dataset 优先用列；否则从文件名解析
    if "software" in df.columns and df["software"].notna().any():
        software = str(df["software"].dropna().iloc[0])
    else:
        software, _ = parse_software_dataset_from_filename(fp)

    if "dataset" in df.columns and df["dataset"].notna().any():
        dataset = str(df["dataset"].dropna().iloc[0])
    else:
        _, dataset = parse_software_dataset_from_filename(fp)

    if software is None or dataset is None:
        raise ValueError(
            f"Cannot determine software/dataset from file '{fp}'. "
            f"Please ensure columns 'software'/'dataset' exist OR filename is Software_Dataset.csv"
        )

    # 找指标列：排除非指标列 + 能转为数值的列
    metric_cols = []
    for c in df.columns:
        if c in NON_METRIC_COLS:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        if s.notna().any():
            metric_cols.append(c)

    if not metric_cols:
        raise ValueError(f"No numeric metric columns found in: {fp}")

    metrics_out = {}
    for m in metric_cols:
        vals = pd.to_numeric(df[m], errors="coerce")
        mean_val = vals.mean()
        sd_val = vals.std(ddof=1)  # 样本标准差
        metrics_out[m] = format_mean_sd(mean_val, sd_val, digits=digits)

    return software, dataset, metrics_out, metric_cols

def main():
    in_dir = os.path.abspath(IN_DIR)
    out_dir = os.path.abspath(OUT_DIR)
    os.makedirs(out_dir, exist_ok=True)

    files = sorted(glob.glob(os.path.join(in_dir, PATTERN)))
    if not files:
        raise FileNotFoundError(f"No files matched: {os.path.join(in_dir, PATTERN)}")

    # dataset -> {software -> {metric -> "mean (sd)"}}
    dataset_map = {}
    # dataset -> set(metrics) 统一列
    dataset_metrics = {}

    for fp in files:
        software, dataset, metrics_dict, metric_cols = summarize_one_file(fp, digits=DIGITS)

        dataset_map.setdefault(dataset, {})
        dataset_map[dataset][software] = metrics_dict

        dataset_metrics.setdefault(dataset, set())
        dataset_metrics[dataset].update(metric_cols)

    # 每个 dataset 输出一张表
    for dataset, software_dict in dataset_map.items():
        metrics = sorted(dataset_metrics[dataset])

        rows = []
        for software, mdict in software_dict.items():
            row = {"software": software}
            for m in metrics:
                row[m] = mdict.get(m, "")
            rows.append(row)

        out_df = pd.DataFrame(rows).set_index("software")
        out_df = out_df.sort_index(key=lambda s: s.str.lower())

        out_path = os.path.join(out_dir, f"{dataset}_mean_sd.csv")
        out_df.to_csv(out_path)

        print(f"[OK] dataset={dataset} -> {out_path}  (rows={out_df.shape[0]}, cols={out_df.shape[1]})")

if __name__ == "__main__":
    main()


[OK] dataset=PBMC -> /root/autodl-tmp/ModuCell/benchmark/final_results/summary/results/PBMC_mean_sd.csv  (rows=7, cols=10)
[OK] dataset=hPancreas -> /root/autodl-tmp/ModuCell/benchmark/final_results/summary/results/hPancreas_mean_sd.csv  (rows=7, cols=10)
[OK] dataset=kidney -> /root/autodl-tmp/ModuCell/benchmark/final_results/summary/results/kidney_mean_sd.csv  (rows=7, cols=10)
[OK] dataset=lung -> /root/autodl-tmp/ModuCell/benchmark/final_results/summary/results/lung_mean_sd.csv  (rows=7, cols=10)


In [2]:
import os
import glob
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon, mannwhitneyu

# ===================== 你只需要改这里 =====================
IN_DIR = "./results"          # 存放 ModuCell_kidney.csv 这类文件的目录
OUT_DIR = "./figures"          # 输出图片的目录
FILE_PATTERN = "*.csv"                    # 默认读所有csv；也可改成 "*_kidney.csv" 等
FIG_DPI = 300
SAVE_FORMAT = "pdf"                       # "png" 或 "pdf"
# =========================================================

SOFTWARE_ORDER = ["scmap_cluster", "scmap_cell", "singleR", "cb", "TOSICA", "scBert", "ModuCell"]
BASELINE_SOFTWARE = "ModuCell"

NON_METRIC_COLS = {"software", "dataset", "ID", "id", "fold", "seed"}

def parse_software_dataset_from_filename(fp: str):
    """从 Software_Dataset.csv 解析软件与数据集（用最后一个 '_' 分割）"""
    base = os.path.basename(fp)
    name = os.path.splitext(base)[0]
    if "_" not in name:
        return None, None
    software, dataset = name.rsplit("_", 1)
    return software, dataset

def stars_from_p(p):
    if p is None or np.isnan(p):
        return ""
    if p < 0.005:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

def get_metric_cols(df: pd.DataFrame):
    metric_cols = []
    for c in df.columns:
        if c in NON_METRIC_COLS:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        if s.notna().any():
            metric_cols.append(c)
    return metric_cols

def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")

def pvalue_vs_baseline(df_a: pd.DataFrame, df_b: pd.DataFrame, metric: str):
    """
    df_a: 某软件数据 (必须有 ID 和 metric)
    df_b: baseline (ModuCell) 数据
    - 若ID可配对：Wilcoxon
    - 否则：Mann-Whitney U
    """
    # 尽量使用 ID 配对
    if "ID" in df_a.columns and "ID" in df_b.columns:
        common = sorted(set(df_a["ID"]) & set(df_b["ID"]))
        if len(common) >= 2:
            xa = safe_numeric(df_a.set_index("ID").loc[common, metric]).dropna()
            xb = safe_numeric(df_b.set_index("ID").loc[common, metric]).dropna()
            common2 = sorted(set(xa.index) & set(xb.index))
            if len(common2) >= 2:
                xa2 = xa.loc[common2].to_numpy()
                xb2 = xb.loc[common2].to_numpy()
                # wilcoxon 要求非全相等且至少2对
                try:
                    stat, p = wilcoxon(xa2, xb2, alternative="two-sided", zero_method="wilcox")
                    return float(p)
                except Exception:
                    # 若全相等等导致失败，则视为不显著
                    return 1.0

    # 非配对：Mann–Whitney U
    xa = safe_numeric(df_a[metric]).dropna().to_numpy()
    xb = safe_numeric(df_b[metric]).dropna().to_numpy()
    if len(xa) >= 2 and len(xb) >= 2:
        try:
            stat, p = mannwhitneyu(xa, xb, alternative="two-sided")
            return float(p)
        except Exception:
            return 1.0

    return np.nan

def load_all_files(in_dir, pattern):
    files = sorted(glob.glob(os.path.join(in_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"No files matched: {os.path.join(in_dir, pattern)}")

    # dataset -> software -> df
    data = {}
    for fp in files:
        df = pd.read_csv(fp)

        # software/dataset：优先列，其次文件名
        if "software" in df.columns and df["software"].notna().any():
            software = str(df["software"].dropna().iloc[0])
        else:
            software, _ = parse_software_dataset_from_filename(fp)

        if "dataset" in df.columns and df["dataset"].notna().any():
            dataset = str(df["dataset"].dropna().iloc[0])
        else:
            _, dataset = parse_software_dataset_from_filename(fp)

        if software is None or dataset is None:
            # 跳过无法解析的文件
            continue

        # 确保有 ID 列（如果没有，就用行号当ID）
        if "ID" not in df.columns:
            df = df.copy()
            df["ID"] = [str(i) for i in range(len(df))]

        data.setdefault(dataset, {})
        data[dataset][software] = df

    return data

def plot_dataset_bars(dataset: str, software_to_df: dict, out_dir: str):
    # baseline 必须存在，否则无法做显著性标注
    if BASELINE_SOFTWARE not in software_to_df:
        print(f"[WARN] dataset={dataset}: baseline '{BASELINE_SOFTWARE}' not found, skip plotting.")
        return

    # 统一指标集合：取所有软件df中能识别的数值指标列的并集
    metrics = set()
    for sw, df in software_to_df.items():
        metrics.update(get_metric_cols(df))
    metrics = sorted(metrics)

    if not metrics:
        print(f"[WARN] dataset={dataset}: no metric columns found.")
        return

    # 子图布局
    n = len(metrics)
    ncols = 3 if n >= 3 else n
    nrows = int(math.ceil(n / ncols))

    # 颜色：用 matplotlib 默认色板，不手动指定具体颜色值
    default_colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", [])
    if not default_colors:
        default_colors = ["C0", "C1", "C2", "C3", "C4", "C5", "C6"]
    color_map = {sw: default_colors[i % len(default_colors)] for i, sw in enumerate(SOFTWARE_ORDER)}

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4.2*ncols, 3.6*nrows), dpi=FIG_DPI)
    axes = np.array(axes).reshape(-1)

    baseline_df = software_to_df[BASELINE_SOFTWARE]

    for idx, metric in enumerate(metrics):
        ax = axes[idx]

        # 计算每个软件的均值
        means = []
        pvals = []
        for sw in SOFTWARE_ORDER:
            if sw in software_to_df:
                df = software_to_df[sw]
                vals = safe_numeric(df[metric]).dropna()
                means.append(float(vals.mean()) if len(vals) > 0 else np.nan)
            else:
                means.append(np.nan)

        # 显著性：非baseline才算
        for sw in SOFTWARE_ORDER:
            if sw == BASELINE_SOFTWARE or sw not in software_to_df:
                pvals.append(np.nan)
            else:
                p = pvalue_vs_baseline(software_to_df[sw], baseline_df, metric)
                pvals.append(p)

        x = np.arange(len(SOFTWARE_ORDER))

        # 画柱子（缺失软件/缺失指标 -> 不画柱）
        for i, sw in enumerate(SOFTWARE_ORDER):
            h = means[i]
            if np.isnan(h):
                continue
            ax.bar(i, h, color=color_map[sw], edgecolor="black", linewidth=0.6)

        ax.set_xticks(x)
        ax.set_xticklabels(SOFTWARE_ORDER, rotation=35, ha="right")
        ax.set_ylabel("score")
        ax.set_title(metric)

        # y 轴上限为 max + margin，方便放星号
        valid = [v for v in means if not np.isnan(v)]
        if valid:
            ymax = max(valid)
            ymin = min(valid)
            margin = (ymax - ymin) * 0.15 if ymax > ymin else (ymax * 0.15 if ymax != 0 else 0.1)
            ax.set_ylim(0 if ymin >= 0 else ymin - margin*0.2, ymax + margin)

        # 标注显著性星号：放在柱子上方
        for i, sw in enumerate(SOFTWARE_ORDER):
            if sw == BASELINE_SOFTWARE:
                continue
            if np.isnan(means[i]):
                continue
            stars = stars_from_p(pvals[i])
            if stars:
                # 星号位置：柱顶 + 2% ~ 5% 的偏移
                y = means[i]
                yoff = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.04
                ax.text(i, y + yoff, stars, ha="center", va="bottom", fontsize=12, fontweight="bold")

        # 细节美化
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    # 多余子图隐藏
    for j in range(len(metrics), len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"{dataset}: metrics by software (mean over IDs)", y=1.02, fontsize=14)
    fig.tight_layout()

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{dataset}_bars.{SAVE_FORMAT}")
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] saved: {out_path}")

def main():
    in_dir = os.path.abspath(IN_DIR)
    out_dir = os.path.abspath(OUT_DIR)

    data = load_all_files(in_dir, FILE_PATTERN)
    if not data:
        raise RuntimeError("No valid (software,dataset) CSV files found.")

    # 每个数据集一张图
    for dataset, software_to_df in data.items():
        # 只保留你关心的软件（按固定顺序）
        filtered = {sw: df for sw, df in software_to_df.items() if sw in SOFTWARE_ORDER}
        plot_dataset_bars(dataset, filtered, out_dir)

if __name__ == "__main__":
    main()


[WARN] dataset=sd: baseline 'ModuCell' not found, skip plotting.


In [18]:
import os, glob, re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon, mannwhitneyu

# ===================== YOU EDIT THESE =====================
IN_DIR = "./"                 # folder containing Software_Dataset.csv files
OUT_DIR = "./figures"
FILE_PATTERN = "*.csv"        # e.g., "*.csv" or "*_kidney.csv"
SAVE_FORMAT = "pdf"           # "png" or "pdf"
DPI = 300
# ==========================================================

SOFTWARE_ORDER = ["scmap_cluster", "scmap_cell", "singleR", "cb", "TOSICA", "scBert", "ModuCell"]
BASELINE = "ModuCell"

# 只画这四个指标（顺序固定）
METRICS_TO_PLOT = ["accuracy", "macro_recall", "macro_precision", "macro_f1"]

# 非指标列名（用于自动识别指标列时排除）
NON_METRIC_COLS = {"software", "dataset", "ID", "id", "fold", "seed"}

# -------- 软件颜色：你可以自行调整（确保每个软件一个颜色）--------
SOFTWARE_COLORS = {
    "scmap_cluster": "#4E79A7",
    "scmap_cell":    "#F28E2B",
    "singleR":       "#E15759",
    "cb":            "#76B7B2",
    "TOSICA":        "#59A14F",
    "scBert":        "#EDC948",
    "ModuCell":      "#B07AA1",
}

def parse_software_dataset_from_filename(fp: str):
    base = os.path.basename(fp)
    name = os.path.splitext(base)[0]
    if "_" not in name:
        return None, None
    software, dataset = name.rsplit("_", 1)
    return software, dataset

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def stars_from_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ""
    if p < 0.005:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

def pvalue_vs_baseline(df_a: pd.DataFrame, df_b: pd.DataFrame, metric: str):
    """
    Prefer paired Wilcoxon by ID if possible.
    Fallback to Mann–Whitney U if cannot pair.
    """
    # Prefer paired test by ID if possible
    if "ID" in df_a.columns and "ID" in df_b.columns and metric in df_a.columns and metric in df_b.columns:
        ida = df_a["ID"].astype(str)
        idb = df_b["ID"].astype(str)
        common = sorted(set(ida) & set(idb))
        if len(common) >= 2:
            a2 = df_a.assign(ID=ida).set_index("ID").loc[common, metric]
            b2 = df_b.assign(ID=idb).set_index("ID").loc[common, metric]
            xa = safe_numeric(a2).dropna()
            xb = safe_numeric(b2).dropna()
            common2 = sorted(set(xa.index) & set(xb.index))
            if len(common2) >= 2:
                xa2 = xa.loc[common2].to_numpy()
                xb2 = xb.loc[common2].to_numpy()
                # Wilcoxon requires not all differences to be zero
                if np.allclose(xa2 - xb2, 0):
                    return 1.0
                try:
                    _, p = wilcoxon(xa2, xb2, alternative="two-sided", zero_method="wilcox")
                    return float(p)
                except Exception:
                    # conservative fallback
                    return 1.0

    # Fallback: Mann–Whitney U
    if metric in df_a.columns and metric in df_b.columns:
        xa = safe_numeric(df_a[metric]).dropna().to_numpy()
        xb = safe_numeric(df_b[metric]).dropna().to_numpy()
        if len(xa) >= 2 and len(xb) >= 2:
            try:
                _, p = mannwhitneyu(xa, xb, alternative="two-sided")
                return float(p)
            except Exception:
                return 1.0
    return np.nan

def load_all(in_dir, pattern):
    files = sorted(glob.glob(os.path.join(in_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"No files matched: {os.path.join(in_dir, pattern)}")

    # dataset -> software -> df
    data = {}
    for fp in files:
        try:
            df = pd.read_csv(fp)
        except Exception:
            continue

        # Determine software/dataset from columns or filename
        software = None
        dataset = None
        if "software" in df.columns and df["software"].notna().any():
            software = str(df["software"].dropna().iloc[0])
        if "dataset" in df.columns and df["dataset"].notna().any():
            dataset = str(df["dataset"].dropna().iloc[0])

        if software is None or dataset is None:
            sw2, ds2 = parse_software_dataset_from_filename(fp)
            software = software or sw2
            dataset = dataset or ds2

        if software is None or dataset is None:
            continue

        # Ensure ID exists; if missing, use row index
        df = df.copy()
        if "ID" not in df.columns:
            df["ID"] = [str(i) for i in range(len(df))]
        else:
            df["ID"] = df["ID"].astype(str)

        data.setdefault(dataset, {})
        data[dataset][software] = df

    return data

def summarize_mean_sd(software_to_df, metrics):
    # returns dict software -> {metric: (mean, sd, n)}
    out = {}
    for sw, df in software_to_df.items():
        out[sw] = {}
        for m in metrics:
            if m not in df.columns:
                out[sw][m] = (np.nan, np.nan, 0)
                continue
            vals = safe_numeric(df[m]).dropna()
            if len(vals) == 0:
                out[sw][m] = (np.nan, np.nan, 0)
            else:
                out[sw][m] = (float(vals.mean()), float(vals.std(ddof=1)), int(len(vals)))
    return out

def plot_dataset(dataset, software_to_df, out_dir):
    if BASELINE not in software_to_df:
        print(f"[WARN] dataset={dataset}: baseline '{BASELINE}' not found; skipping.")
        return None

    # filter only target softwares (keep if present)
    software_to_df = {sw: df for sw, df in software_to_df.items() if sw in SOFTWARE_ORDER}
    if not software_to_df:
        return None

    # 只用指定的四个指标（存在才画；缺失的子图会提示并空着）
    metrics = METRICS_TO_PLOT

    # 固定横向一排四张
    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(4.8*4, 4.2), dpi=DPI)
    axes = np.array(axes).reshape(-1)

    stats = summarize_mean_sd(software_to_df, metrics)
    baseline_df = software_to_df[BASELINE]

    for i, metric in enumerate(metrics):
        ax = axes[i]
        xs = np.arange(len(SOFTWARE_ORDER))

        heights = []
        errs = []
        pvals = []

        for sw in SOFTWARE_ORDER:
            if sw not in stats:
                heights.append(np.nan)
                errs.append(np.nan)
                pvals.append(np.nan)
                continue

            mean, sd, nobs = stats[sw].get(metric, (np.nan, np.nan, 0))
            heights.append(mean)
            errs.append(sd)

            if sw == BASELINE:
                pvals.append(np.nan)
            else:
                if (sw in software_to_df) and (metric in software_to_df[sw].columns) and (metric in baseline_df.columns):
                    pvals.append(pvalue_vs_baseline(software_to_df[sw], baseline_df, metric))
                else:
                    pvals.append(np.nan)

        # bars with per-software colors
        for j, sw in enumerate(SOFTWARE_ORDER):
            if np.isnan(heights[j]):
                continue
            color = SOFTWARE_COLORS.get(sw, None)
            ax.bar(
                j, heights[j],
                yerr=errs[j] if not np.isnan(errs[j]) else None,
                capsize=3,
                edgecolor="black",
                linewidth=0.6,
                color=color
            )

        ax.set_xticks(xs)
        ax.set_xticklabels(SOFTWARE_ORDER, rotation=35, ha="right")
        ax.set_ylabel("mean score")
        ax.set_title(metric)

        # y-lim with margin
        valid = [h for h in heights if not np.isnan(h)]
        if valid:
            ymax = max(valid)
            ymin = min(valid)
            margin = (ymax - ymin) * 0.2 if ymax > ymin else (abs(ymax) * 0.2 if ymax != 0 else 0.1)
            lower = 0 if ymin >= 0 else ymin - margin*0.2
            ax.set_ylim(lower, ymax + margin)

        # significance stars over non-baseline bars
        for j, sw in enumerate(SOFTWARE_ORDER):
            if sw == BASELINE or np.isnan(heights[j]):
                continue
            st = stars_from_p(pvals[j])
            if not st:
                continue
            y = heights[j]
            yerr = errs[j] if not np.isnan(errs[j]) else 0.0
            yoff = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.04
            ax.text(j, y + yerr + yoff, st, ha="center", va="bottom", fontsize=12, fontweight="bold")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(f"{dataset}: mean ± SD over folds (paired vs {BASELINE} if possible)", y=1.05, fontsize=14)
    fig.tight_layout()

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{dataset}_bars.{SAVE_FORMAT}")
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] saved: {out_path}")
    return out_path

# ----- run -----
data = load_all(IN_DIR, FILE_PATTERN)

saved = []
for dataset, sw_map in data.items():
    p = plot_dataset(dataset, sw_map, OUT_DIR)
    if p:
        saved.append(p)

saved


[OK] saved: ./figures/PBMC_bars.pdf
[OK] saved: ./figures/hPancreas_bars.pdf
[OK] saved: ./figures/kidney_bars.pdf
[OK] saved: ./figures/lung_bars.pdf


['./figures/PBMC_bars.pdf',
 './figures/hPancreas_bars.pdf',
 './figures/kidney_bars.pdf',
 './figures/lung_bars.pdf']

In [21]:
import os, glob, re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon, mannwhitneyu

# ===================== YOU EDIT THESE =====================
IN_DIR = "./"                 # folder containing Software_Dataset.csv files
OUT_DIR = "./figures"
FILE_PATTERN = "*.csv"        # e.g., "*.csv" or "*_kidney.csv"
SAVE_FORMAT = "pdf"           # "png" or "pdf"
DPI = 300
# ==========================================================

SOFTWARE_ORDER = ["scmap_cluster", "scmap_cell", "singleR", "cb", "TOSICA", "scBert", "ModuCell"]
BASELINE = "ModuCell"

# 只画这四个指标（顺序固定）
METRICS_TO_PLOT = ["accuracy", "macro_recall", "macro_precision", "macro_f1"]

# 非指标列名（用于自动识别指标列时排除）
NON_METRIC_COLS = {"software", "dataset", "ID", "id", "fold", "seed"}

# -------- 软件颜色：你可以自行调整（确保每个软件一个颜色）--------
SOFTWARE_COLORS = {
    "scmap_cluster": "#4E79A7",
    "scmap_cell":    "#F28E2B",
    "singleR":       "#E15759",
    "cb":            "#76B7B2",
    "TOSICA":        "#59A14F",
    "scBert":        "#EDC948",
    "ModuCell":      "#B07AA1",
}

def parse_software_dataset_from_filename(fp: str):
    base = os.path.basename(fp)
    name = os.path.splitext(base)[0]
    if "_" not in name:
        return None, None
    software, dataset = name.rsplit("_", 1)
    return software, dataset

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def stars_from_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ""
    if p < 0.005:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

def pvalue_vs_baseline(df_a: pd.DataFrame, df_b: pd.DataFrame, metric: str):
    """
    Prefer paired Wilcoxon by ID if possible.
    Fallback to Mann–Whitney U if cannot pair.
    """
    # Prefer paired test by ID if possible
    if "ID" in df_a.columns and "ID" in df_b.columns and metric in df_a.columns and metric in df_b.columns:
        ida = df_a["ID"].astype(str)
        idb = df_b["ID"].astype(str)
        common = sorted(set(ida) & set(idb))
        if len(common) >= 2:
            a2 = df_a.assign(ID=ida).set_index("ID").loc[common, metric]
            b2 = df_b.assign(ID=idb).set_index("ID").loc[common, metric]
            xa = safe_numeric(a2).dropna()
            xb = safe_numeric(b2).dropna()
            common2 = sorted(set(xa.index) & set(xb.index))
            if len(common2) >= 2:
                xa2 = xa.loc[common2].to_numpy()
                xb2 = xb.loc[common2].to_numpy()
                # Wilcoxon requires not all differences to be zero
                if np.allclose(xa2 - xb2, 0):
                    return 1.0
                try:
                    _, p = wilcoxon(xa2, xb2, alternative="two-sided", zero_method="wilcox")
                    return float(p)
                except Exception:
                    return 1.0

    # Fallback: Mann–Whitney U
    if metric in df_a.columns and metric in df_b.columns:
        xa = safe_numeric(df_a[metric]).dropna().to_numpy()
        xb = safe_numeric(df_b[metric]).dropna().to_numpy()
        if len(xa) >= 2 and len(xb) >= 2:
            try:
                _, p = mannwhitneyu(xa, xb, alternative="two-sided")
                return float(p)
            except Exception:
                return 1.0
    return np.nan

def load_all(in_dir, pattern):
    files = sorted(glob.glob(os.path.join(in_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"No files matched: {os.path.join(in_dir, pattern)}")

    # dataset -> software -> df
    data = {}
    for fp in files:
        try:
            df = pd.read_csv(fp)
        except Exception:
            continue

        # Determine software/dataset from columns or filename
        software = None
        dataset = None
        if "software" in df.columns and df["software"].notna().any():
            software = str(df["software"].dropna().iloc[0])
        if "dataset" in df.columns and df["dataset"].notna().any():
            dataset = str(df["dataset"].dropna().iloc[0])

        if software is None or dataset is None:
            sw2, ds2 = parse_software_dataset_from_filename(fp)
            software = software or sw2
            dataset = dataset or ds2

        if software is None or dataset is None:
            continue

        # Ensure ID exists; if missing, use row index
        df = df.copy()
        if "ID" not in df.columns:
            df["ID"] = [str(i) for i in range(len(df))]
        else:
            df["ID"] = df["ID"].astype(str)

        data.setdefault(dataset, {})
        data[dataset][software] = df

    return data

def summarize_mean_sd(software_to_df, metrics):
    # returns dict software -> {metric: (mean, sd, n)}
    out = {}
    for sw, df in software_to_df.items():
        out[sw] = {}
        for m in metrics:
            if m not in df.columns:
                out[sw][m] = (np.nan, np.nan, 0)
                continue
            vals = safe_numeric(df[m]).dropna()
            if len(vals) == 0:
                out[sw][m] = (np.nan, np.nan, 0)
            else:
                out[sw][m] = (float(vals.mean()), float(vals.std(ddof=1)), int(len(vals)))
    return out

def plot_dataset(dataset, software_to_df, out_dir):
    if BASELINE not in software_to_df:
        print(f"[WARN] dataset={dataset}: baseline '{BASELINE}' not found; skipping.")
        return None

    # filter only target softwares (keep if present)
    software_to_df = {sw: df for sw, df in software_to_df.items() if sw in SOFTWARE_ORDER}
    if not software_to_df:
        return None

    metrics = METRICS_TO_PLOT

    # 固定横向一排四张
    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(4.8*4, 4.2), dpi=DPI)
    axes = np.array(axes).reshape(-1)

    stats = summarize_mean_sd(software_to_df, metrics)
    baseline_df = software_to_df[BASELINE]

    for i, metric in enumerate(metrics):
        ax = axes[i]
        xs = np.arange(len(SOFTWARE_ORDER))

        heights = []
        errs = []
        pvals = []

        for sw in SOFTWARE_ORDER:
            if sw not in stats:
                heights.append(np.nan)
                errs.append(np.nan)
                pvals.append(np.nan)
                continue

            mean, sd, nobs = stats[sw].get(metric, (np.nan, np.nan, 0))
            heights.append(mean)
            errs.append(sd)

            if sw == BASELINE:
                pvals.append(np.nan)
            else:
                if (sw in software_to_df) and (metric in software_to_df[sw].columns) and (metric in baseline_df.columns):
                    pvals.append(pvalue_vs_baseline(software_to_df[sw], baseline_df, metric))
                else:
                    pvals.append(np.nan)

        # bars with per-software colors
        for j, sw in enumerate(SOFTWARE_ORDER):
            if np.isnan(heights[j]):
                continue
            color = SOFTWARE_COLORS.get(sw, None)
            ax.bar(
                j, heights[j],
                yerr=errs[j] if not np.isnan(errs[j]) else None,
                capsize=3,
                edgecolor="black",
                linewidth=0.6,
                color=color
            )

        ax.set_xticks(xs)
        ax.set_xticklabels(SOFTWARE_ORDER, rotation=35, ha="right")
        ax.set_ylabel("mean score")
        ax.set_title(metric)

        # ---- 关键修改：所有子图固定纵坐标 0~1 ----
        ax.set_ylim(0, 1)

        # significance stars over non-baseline bars
        # y轴固定后，用固定偏移更稳；同时避免超过1
        for j, sw in enumerate(SOFTWARE_ORDER):
            if sw == BASELINE or np.isnan(heights[j]):
                continue
            st = stars_from_p(pvals[j])
            if not st:
                continue
            y = heights[j]
            yerr = errs[j] if not np.isnan(errs[j]) else 0.0
            yoff = 0.03  # 相对于0~1的固定偏移
            y_text = min(y + yerr + yoff, 0.98)
            ax.text(j, y_text, st, ha="center", va="bottom", fontsize=12, fontweight="bold")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(f"{dataset}: mean ± SD over folds (paired vs {BASELINE} if possible)", y=1.05, fontsize=14)
    fig.tight_layout()

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{dataset}_bars.{SAVE_FORMAT}")
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] saved: {out_path}")
    return out_path

# ----- run -----
data = load_all(IN_DIR, FILE_PATTERN)

saved = []
for dataset, sw_map in data.items():
    p = plot_dataset(dataset, sw_map, OUT_DIR)
    if p:
        saved.append(p)

saved


[OK] saved: ./figures/PBMC_bars.pdf
[OK] saved: ./figures/hPancreas_bars.pdf
[OK] saved: ./figures/kidney_bars.pdf
[OK] saved: ./figures/lung_bars.pdf


['./figures/PBMC_bars.pdf',
 './figures/hPancreas_bars.pdf',
 './figures/kidney_bars.pdf',
 './figures/lung_bars.pdf']

In [32]:
##0125
import os, glob, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# ===================== YOU EDIT THESE =====================
IN_DIR = "./"                 # folder containing Software_Dataset.csv files
OUT_DIR = "./figures"
FILE_PATTERN = "*.csv"        # e.g., "*.csv" or "*_kidney.csv"
SAVE_FORMAT = "pdf"           # "png" or "pdf"
DPI = 300
# ==========================================================

# 你想要的绘图顺序
SOFTWARE_ORDER = ["scmap_cluster", "scmap_cell", "singleR", "cb", "TOSICA", "scBert", "ModuCell"]
BASELINE = "ModuCell"

# 只画这四个指标（顺序固定）
METRICS_TO_PLOT = ["accuracy", "macro_recall", "macro_precision", "macro_f1"]

# -------- 你可以在图中自定义显示名--------
SOFTWARE_DISPLAY = {
    "scmap_cluster": "scmap_cluster",
    "scmap_cell":    "scmap_cell",
    "singleR":       "SingleR",
    "cb":            "Cell BLAST",
    "TOSICA":        "TOSICA",
    "scBert":        "scBERT",
    "ModuCell":      "ModuCell",
}

# -------- 你可以在图中自定义柱体颜色（fill）--------
BAR_FILL_COLORS = {
    "scmap_cluster": "#bccbe8",
    "scmap_cell":    "#c6dfb8",
    "singleR":       "#fcd5b4",
    "cb":            "#d7cae4",
    "TOSICA":        "#e8e7bb",
    "scBert":        "#d5ebf0",
    "ModuCell":      "#FBDFE2",
}

# -------- 你可以在图中自定义边框颜色（edge）--------
BAR_EDGE_COLORS = {
    "scmap_cluster": "#1d73b6",
    "scmap_cell":    "#24a645",
    "singleR":       "#f27830",
    "cb":            "#8768a6",
    "TOSICA":        "#bdbf32",
    "scBert":        "#6dc2d2",
    "ModuCell":      "#B83945",
}

# 如果你想所有柱子边框统一一种颜色，把下面改成例如 "#000000"
EDGE_COLOR_FALLBACK = ""
EDGE_LINEWIDTH = 3

# -------- 显著性星号规则 --------
def stars_from_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ""
    if p < 0.001:
        return "****"
    if p < 0.005:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def parse_software_dataset_from_filename(fp: str):
    """
    推荐文件名：{software}_{dataset}.csv 或 {software}_{dataset}_*.csv
    为避免 dataset 中含 '_' 的歧义，这里优先匹配 SOFTWARE_ORDER 作为前缀。
    """
    base = os.path.basename(fp)
    name = os.path.splitext(base)[0]

    for sw in SOFTWARE_ORDER:
        prefix = sw + "_"
        if name.startswith(prefix):
            rest = name[len(prefix):]
            # 若存在额外后缀（如 _v2），默认取第一个段作为 dataset
            # 若你的 dataset 本身含 '_'，建议在CSV里提供 dataset 列
            dataset = rest.split("_")[0] if "_" in rest else rest
            return sw, dataset

    if "_" not in name:
        return None, None
    sw, ds = name.rsplit("_", 1)
    return sw, ds

def load_all(in_dir, pattern):
    files = sorted(glob.glob(os.path.join(in_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"No files matched: {os.path.join(in_dir, pattern)}")

    # dataset -> software -> df
    data = {}
    for fp in files:
        try:
            df = pd.read_csv(fp)
        except Exception:
            continue

        software = None
        dataset = None

        # 优先使用表内列（如果有）
        if "software" in df.columns and df["software"].notna().any():
            software = str(df["software"].dropna().iloc[0])
        if "dataset" in df.columns and df["dataset"].notna().any():
            dataset = str(df["dataset"].dropna().iloc[0])

        # 否则从文件名推断
        if software is None or dataset is None:
            sw2, ds2 = parse_software_dataset_from_filename(fp)
            software = software or sw2
            dataset = dataset or ds2

        if software is None or dataset is None:
            continue

        df = df.copy()
        # ttest_ind 不需要配对ID，但保留ID方便检查
        if "ID" not in df.columns:
            df["ID"] = [str(i) for i in range(len(df))]
        else:
            df["ID"] = df["ID"].astype(str)

        data.setdefault(dataset, {})
        data[dataset][software] = df

    return data

def summarize_mean_sd(software_to_df, metrics):
    # returns dict software -> {metric: (mean, sd, n)}
    out = {}
    for sw, df in software_to_df.items():
        out[sw] = {}
        for m in metrics:
            if m not in df.columns:
                out[sw][m] = (np.nan, np.nan, 0)
                continue
            vals = safe_numeric(df[m])
            vals = vals[np.isfinite(vals)]
            if len(vals) == 0:
                out[sw][m] = (np.nan, np.nan, 0)
            else:
                sd = float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else 0.0
                out[sw][m] = (float(np.nanmean(vals)), sd, int(len(vals)))
    return out

def ttest_pvalue(ref_vals, other_vals):
    """
    按你要求：_, p = ttest_ind(ref_vals, other_vals, equal_var=False, nan_policy="omit")
    (Welch t-test)
    """
    ref_vals = safe_numeric(pd.Series(ref_vals))
    other_vals = safe_numeric(pd.Series(other_vals))
    _, p = ttest_ind(ref_vals, other_vals, equal_var=False, nan_policy="omit")
    return float(p) if p is not None else np.nan

def plot_dataset(dataset, software_to_df, out_dir):
    if BASELINE not in software_to_df:
        print(f"[WARN] dataset={dataset}: baseline '{BASELINE}' not found; skipping.")
        return None

    # 只保留目标软件
    software_to_df = {sw: df for sw, df in software_to_df.items() if sw in SOFTWARE_ORDER}
    if not software_to_df:
        return None

    metrics = METRICS_TO_PLOT

    # 固定横向一排四张
    fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(4.9*4, 4.3), dpi=DPI)
    axes = np.array(axes).reshape(-1)

    stats = summarize_mean_sd(software_to_df, metrics)
    ref_df = software_to_df[BASELINE]

    # -------- 同一数据集统一 y 轴范围 --------
    global_max = 0.0
    for m in metrics:
        for sw in SOFTWARE_ORDER:
            if sw in stats and m in stats[sw]:
                mean, sd, _ = stats[sw][m]
                if np.isfinite(mean):
                    sd = sd if np.isfinite(sd) else 0.0
                    global_max = max(global_max, mean + sd)
    y_upper = min(1.0, global_max + 0.08) if global_max > 0 else 1.0
    y_lower = 0.0

    for i, metric in enumerate(metrics):
        ax = axes[i]
        xs = np.arange(len(SOFTWARE_ORDER))

        heights, errs, pvals = [], [], []

        ref_vals = safe_numeric(ref_df[metric]) if metric in ref_df.columns else pd.Series([], dtype=float)

        for sw in SOFTWARE_ORDER:
            if sw not in stats:
                heights.append(np.nan)
                errs.append(np.nan)
                pvals.append(np.nan)
                continue

            mean, sd, _ = stats[sw].get(metric, (np.nan, np.nan, 0))
            heights.append(mean)
            errs.append(sd)

            if sw == BASELINE:
                pvals.append(np.nan)
            else:
                if sw in software_to_df and metric in software_to_df[sw].columns and metric in ref_df.columns:
                    other_vals = safe_numeric(software_to_df[sw][metric])
                    pvals.append(ttest_pvalue(ref_vals, other_vals))
                else:
                    pvals.append(np.nan)

        # bars with per-software fill + edge colors
        for j, sw in enumerate(SOFTWARE_ORDER):
            if np.isnan(heights[j]):
                continue
            fill = BAR_FILL_COLORS.get(sw, None)
            edge = BAR_EDGE_COLORS.get(sw, EDGE_COLOR_FALLBACK)
            ax.bar(
                j,
                heights[j],
                yerr=errs[j] if not np.isnan(errs[j]) else None,
                capsize=3,
                facecolor=fill,
                edgecolor=edge,
                linewidth=EDGE_LINEWIDTH
            )

        # x labels：用 display 名
        xlabels = [SOFTWARE_DISPLAY.get(sw, sw) for sw in SOFTWARE_ORDER]
        ax.set_xticks(xs)
        ax.set_xticklabels(xlabels, rotation=35, ha="right")

        ax.set_ylabel("mean score")
        ax.set_title(metric)

        ax.set_ylim(y_lower, y_upper)

        # significance stars over non-baseline bars
        for j, sw in enumerate(SOFTWARE_ORDER):
            if sw == BASELINE or np.isnan(heights[j]):
                continue
            st = stars_from_p(pvals[j])
            if not st:
                continue
            y = heights[j]
            yerr = errs[j] if not np.isnan(errs[j]) else 0.0
            yoff = (y_upper - y_lower) * 0.04
            y_text = min(y + yerr + yoff, y_upper - 0.02 * (y_upper - y_lower))
            ax.text(j, y_text, st, ha="center", va="bottom", fontsize=12, fontweight="bold")

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(f"{dataset}: mean ± SD over folds (Welch t-test vs {BASELINE})", y=1.05, fontsize=14)
    fig.tight_layout()

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{dataset}_bars.{SAVE_FORMAT}")
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] saved: {out_path}")
    return out_path

# ----- run -----
data = load_all(IN_DIR, FILE_PATTERN)

saved = []
for dataset, sw_map in data.items():
    p = plot_dataset(dataset, sw_map, OUT_DIR)
    if p:
        saved.append(p)

saved


[OK] saved: ./figures/PBMC_bars.pdf
[OK] saved: ./figures/hPancreas_bars.pdf
[OK] saved: ./figures/kidney_bars.pdf
[OK] saved: ./figures/lung_bars.pdf


['./figures/PBMC_bars.pdf',
 './figures/hPancreas_bars.pdf',
 './figures/kidney_bars.pdf',
 './figures/lung_bars.pdf']